In [ ]:
import numpy as np
import pandas as pd

students = np.random.randint(50, high=101, size=(50, 5))
students_df = pd.DataFrame(students, index=[f"Student#{i}" for i in range(1,51)], columns=['Math', 'Science', 'English', 'History', 'Art'])
students_df

In [15]:
students_df['Total'] = np.sum(students_df.iloc[:, :5],axis=1)
#students_df['Average'] = students_df['Total'] / 5
students_df['Average'] = np.mean(students_df.iloc[:, :5], axis=1)
students_df

,Math,Science,English,History,Art,Total,Average
Student#1,94,55,82,68,73,372,74.4
Student#2,94,96,92,77,52,411,82.2
Student#3,51,55,77,57,87,327,65.4
Student#4,56,59,56,58,73,302,60.4
Student#5,57,95,73,96,58,379,75.8
Student#6,56,90,51,51,90,338,67.6
Student#7,100,57,58,84,59,358,71.6
Student#8,67,57,54,67,55,300,60.0
Student#9,62,60,66,70,96,354,70.8
Student#10,59,71,65,91,55,341,68.2


In [22]:
import pandas as pd
import requests
from io import StringIO

def fetch_twse_monthly_revenue(year: int, month: int, output_csv: str | None = None) -> pd.DataFrame:
    """
    抓取 MOPS 上市公司每月營收（CSV）
    year: 民國年，例如 111
    month: 1~12
    """

    # MOPS 有時候對路徑/檔名有差異，這裡做多候選嘗試
    candidates = [
        f"https://mops.twse.com.tw/nas/t21/sii/t21sc03_{year}_{month}_0.csv",
        f"https://mops.twse.com.tw/nas/t21/sii/t21sc03_{year}_{month}.csv",
        f"https://mops.twse.com.tw/nas/t21/sii/t21sc03_{year}_{month}_0.html",  # 最後才退回 html
    ]

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
        "Referer": "https://mops.twse.com.tw/mops/web/t21sc03",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    last_error = None

    for url in candidates:
        try:
            r = requests.get(url, headers=headers, timeout=20)
            # 直接把狀態碼打印出來，避免你只看到「下載失敗」
            if r.status_code != 200:
                last_error = f"HTTP {r.status_code} for {url}\n{r.text[:200]}"
                continue

            # MOPS 多數是 Big5
            r.encoding = "big5"

            text = r.text

            # 如果拿到的是 html（不是 csv），就提示你改用 csv 候選
            if "<html" in text.lower():
                last_error = f"Got HTML instead of CSV for {url} (可能需要改用 csv 連結)"
                continue

            df = pd.read_csv(StringIO(text)).dropna(how="all")

            if output_csv:
                df.to_csv(output_csv, index=False, encoding="utf-8-sig")

            return df

        except Exception as e:
            last_error = f"{type(e).__name__}: {e} for {url}"
            continue

    raise Exception(f"下載失敗（已嘗試多個網址）\n最後錯誤：{last_error}")